# SLB CDP Climate Change Thematic Coding

This notebook performs a longitudinal thematic analysis of SLB CDP Climate Change reports using PyMuPDF, pandas, matplotlib, numpy, and wordcloud.

The workflow: read PDF reports, extract paragraphs, clean text, assign multiple themes per paragraph, and save the coded output for reporting.

In [ ]:
from __future__ import annotations

import re
from collections import Counter
from pathlib import Path

import fitz
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
from wordcloud import WordCloud


# Default working directory for the notebook.
# Change this to the folder containing the six PDF files if needed.
SOURCE_DIR = Path.cwd()
OUTPUT_EXCEL = SOURCE_DIR / 'slb_cdp_thematic_coding.xlsx'
OUTPUT_CSV = SOURCE_DIR / 'slb_cdp_thematic_coding.csv'
OUTPUT_DIR = SOURCE_DIR

THEMES = {
    'Climate Governance': [
        'board', 'governance', 'committee', 'executive', 'oversight', 'management', 'responsibility'
    ],
    'Climate Strategy': [
        'strategy', 'transition', 'roadmap', 'resilience', 'ambition', 'decarbonization', 'net zero'
    ],
    'Physical Climate Risks': [
        'flood', 'storm', 'hurricane', 'cyclone', 'drought', 'wildfire', 'heatwave', 'sea level', 'extreme weather'
    ],
    'Transition Risks': [
        'policy', 'regulation', 'carbon pricing', 'emissions trading', 'legislation', 'tax', 'compliance'
    ],
    'GHG Emissions': [
        'scope 1', 'scope 2', 'scope 3', 'ghg', 'greenhouse gas', 'methane', 'emissions'
    ],
    'Energy Management': [
        'renewable', 'electricity', 'energy efficiency', 'energy consumption', 'fuel', 'power'
    ],
    'Technology & Innovation': [
        'ccs', 'ccus', 'hydrogen', 'geothermal', 'digital', 'innovation', 'carbon capture', 'electrification'
    ],
    'Operational Efficiency': [
        'optimization', 'efficiency', 'electrification', 'methane reduction', 'operational excellence'
    ],
    'Supply Chain': [
        'supplier', 'procurement', 'logistics', 'value chain', 'contractor'
    ],
    'Climate Opportunities': [
        'opportunity', 'investment', 'growth', 'market', 'customer', 'low carbon'
    ],
    'Risk Management': [
        'risk assessment', 'mitigation', 'adaptation', 'resilience', 'enterprise risk'
    ],
    'Targets & Performance': [
        'target', 'reduction', 'intensity', 'progress', 'net zero', 'baseline'
    ],
    'Sustainable Finance': [
        'capex', 'opex', 'investment', 'financial impact', 'internal carbon price'
    ],
    'Stakeholder Engagement': [
        'stakeholder', 'customer', 'investor', 'collaboration', 'partnership'
    ],
    'Reporting & Transparency': [
        'cdp', 'tcfd', 'ifrs s2', 'gri', 'sasb', 'disclosure', 'assurance', 'transparency'
    ],
}

In [3]:
def discover_reports(directory: Path) -> list[Path]:
    """Return a sorted list of PDF files found in the target directory.
    If no PDFs are found, raise an informative error.
    """
    pdf_files = sorted(directory.glob('*.pdf'), key=lambda path: path.name.lower())
    if not pdf_files:
        raise FileNotFoundError(f'No PDF reports found in {directory}. Place the six CDP PDFs in this folder.')
    return pdf_files

def extract_year_from_filename(file_path: Path) -> str:
    match = re.search(r'(20\d{2})', file_path.name)
    return match.group(1) if match else 'Unknown'

def extract_text_from_pdf(pdf_path: Path) -> str:
    """Read a PDF and return raw text with page breaks preserved.
    PyMuPDF may return text with soft line breaks, so we preserve page boundaries.
    """
    text_parts: list[str] = []
    with fitz.open(pdf_path) as document:
        for page in document:
            text_parts.append(page.get_text('text'))
    return "\n\n".join(text_parts)

def split_into_paragraphs(raw_text: str) -> list[str]:
    """Split text into paragraphs using blank lines as natural separators.
    This keeps paragraph structure while removing empty blocks.
    """
    normalized = raw_text.replace("\u2028", "\n").replace("\u2029", "\n")
    normalized = re.sub(r"\n{2,}", "\n\n", normalized)
    paragraph_chunks = re.split(r"\n\s*\n+", normalized)
    paragraphs = [chunk.strip() for chunk in paragraph_chunks if chunk.strip()]
    return paragraphs

def clean_text(paragraph: str) -> str:
    """Lowercase text, remove punctuation, normalize whitespace, and keep numbers.
    Scope references such as 'Scope 1' remain intact.
    """
    cleaned = paragraph.lower()
    cleaned = cleaned.replace('–', ' ').replace('—', ' ')
    cleaned = re.sub(r'[^\w\s]', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

def compile_theme_patterns(theme_dictionary: dict[str, list[str]]) -> dict[str, list[re.Pattern]]:
    compiled: dict[str, list[re.Pattern]] = {}
    for theme, keywords in theme_dictionary.items():
        compiled[theme] = [
            re.compile(r'\b' + re.escape(keyword) + r'\b') for keyword in keywords
        ]
    return compiled

def assign_themes_to_paragraph(paragraph: str, patterns: dict[str, list[re.Pattern]]) -> list[str]:
    return [
        theme
        for theme, regex_list in patterns.items()
        if any(regex.search(paragraph) for regex in regex_list)
    ]

In [23]:
def build_coding_dataframe(pdf_paths: list[Path]) -> pd.DataFrame:
    """Build a dataframe of coded paragraphs for all PDF reports.
    Each paragraph may map to multiple themes.
    """
    theme_patterns = compile_theme_patterns(THEMES)
    rows: list[dict[str, object]] = []

    for pdf_path in pdf_paths:
        year = extract_year_from_filename(pdf_path)
        raw_text = extract_text_from_pdf(pdf_path)
        paragraphs = split_into_paragraphs(raw_text)

        for index, paragraph in enumerate(paragraphs, start=1):
            cleaned_paragraph = clean_text(paragraph)
            if not cleaned_paragraph:
                continue

            themes = assign_themes_to_paragraph(cleaned_paragraph, theme_patterns)
            paragraph_id = f'{year}-{index:04d}'

            rows.append({
                'Year': year,
                'Paragraph_ID': paragraph_id,
                'Paragraph_Text': cleaned_paragraph,
                'Assigned_Themes': themes,
            })

    dataframe = pd.DataFrame(rows)
    dataframe['Assigned_Themes'] = dataframe['Assigned_Themes'].apply(lambda x: ', '.join(x) if x else '')
    return dataframe

def save_dataframe(dataframe: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_excel(output_path, index=False, engine='openpyxl')
    dataframe.to_csv(output_path.with_suffix('.csv'), index=False)
    print(f'Saved thematic coding to: {output_path}')
    print(f'Also saved CSV export to: {output_path.with_suffix(".csv")}')

def plot_theme_distribution(dataframe: pd.DataFrame, output_dir: Path) -> None:
    """Plot theme frequency by year for longitudinal analysis."""
    if dataframe.empty:
        print('No coded paragraphs were generated; skipping visualization.')
        return

    analysis_df = dataframe.copy()
    analysis_df['Year'] = pd.to_numeric(analysis_df['Year'], errors='coerce')
    analysis_df = analysis_df.dropna(subset=['Year']).copy()

    if analysis_df.empty:
        print('No valid year values were found; skipping visualization.')
        return

    exploded_df = (
        analysis_df.assign(Themes=analysis_df['Assigned_Themes'].fillna('').str.split(', '))
        .explode('Themes')
        .rename(columns={'Themes': 'Theme'})
    )
    exploded_df = exploded_df[exploded_df['Theme'] != '']

    if exploded_df.empty:
        print('No theme assignments were generated; skipping visualization.')
        return

    year_theme_counts = (
        exploded_df.groupby(['Year', 'Theme'])
        .size()
        .reset_index(name='Count')
    )

    if year_theme_counts.empty:
        print('No theme assignments were generated; skipping visualization.')
        return

    pivot_df = year_theme_counts.pivot(index='Theme', columns='Year', values='Count').fillna(0)
    pivot_df = pivot_df.sort_index()

    fig, ax = plt.subplots(figsize=(14, 7))
    colors = plt.cm.Set2(range(len(pivot_df.columns)))
    for column, color in zip(pivot_df.columns, colors):
        ax.barh(pivot_df.index, pivot_df[column], color=color, label=str(int(column)))

    ax.set_title('SLB CDP Report Theme Frequency by Year')
    ax.set_xlabel('Count')
    ax.set_ylabel('Theme')
    ax.grid(axis='x', linestyle='--', alpha=0.35)
    ax.legend(title='Year', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    chart_path = output_dir / 'theme_frequency.png'
    fig.savefig(chart_path, dpi=150)
    plt.close(fig)
    print(f'Saved theme distribution chart to: {chart_path}')

    corpus = ' '.join(analysis_df['Paragraph_Text'].astype(str).tolist())
    wordcloud = WordCloud(
        width=1200,
        height=600,
        background_color='white',
        collocations=False,
        max_words=200,
    ).generate(corpus)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis('off')
    cloud_path = output_dir / 'cdp_wordcloud.png'
    fig.savefig(cloud_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved word cloud to: {cloud_path}')

In [24]:
pdf_reports = discover_reports(SOURCE_DIR)
print('Found PDF reports:')
for pdf in pdf_reports:
    print(f'- {pdf.name}')

coding_df = build_coding_dataframe(pdf_reports)
print(f'Processed paragraphs: {len(coding_df)}')

save_dataframe(coding_df, OUTPUT_EXCEL)
plot_theme_distribution(coding_df, OUTPUT_DIR)

coding_df.head(10)

Found PDF reports:
- 2021-cdp-climate-change.pdf
- 2023-cdp-sc.pdf
- cdp-climate-change-2024.pdf
- cdp-climate-change-2025.pdf
Processed paragraphs: 2022
Saved thematic coding to: e:\ESG\SLB\slb_cdp_thematic_coding.xlsx
Also saved CSV export to: e:\ESG\SLB\slb_cdp_thematic_coding.csv
Saved theme distribution chart to: e:\ESG\SLB\theme_frequency.png


NameError: name 'WordCloud' is not defined

## Notes for report generation

- The generated CSV file can serve as the basis for tables, trend charts, and longitudinal analysis.
- Each paragraph preserves cleaned text and assigned theme labels.
- Use `Assigned_Themes` aggregation and year grouping to compare topic frequency over time.